In [ ]:
! pip install autogluon

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 13.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 28.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime.

In [ ]:
import sys, os
import matplotlib
import time
import pandas as pd
import numpy
import ast
import json
import matplotlib.pyplot as plt
matplotlib.use('Agg')
from sklearn.neighbors import KNeighborsClassifier as knnbase
from sklearn.ensemble import RandomForestClassifier as rf
from sklearn.naive_bayes import MultinomialNB as mnb
from sklearn.linear_model import LogisticRegression as LR
from sklearn.naive_bayes import GaussianNB as GNB

from autogluon.tabular import TabularDataset
from autogluon.tabular import TabularPredictor as task
from autogluon.core.utils import infer_problem_type

from sklearn.model_selection import GridSearchCV as GSCV
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import MinMaxScaler as MMS
from sklearn.preprocessing import StandardScaler as SS

from sklearn.metrics import accuracy_score, hamming_loss, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import multilabel_confusion_matrix as ML_matrix
from sklearn.metrics import precision_recall_fscore_support as score_multi
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from pickle import load, dump
from sklearn.utils import resample
from sklearn.metrics import classification_report
from scipy.stats import ttest_ind


In [ ]:
# -------------------------------- HELPERS ------------------------------------------ #
def split_df(Xdata, labels, testsplit=0.3):
	Xtrain,Xtest,ytrain,ytest = train_test_split(Xdata,labels,test_size=testsplit)
	return Xtrain, Xtest, ytrain, ytest

# Rescale values to fit in a range; default: 0-1
def normalize(Xtrain, Xtest):
	scaler = MMS(feature_range=(0,1))
	Xtrainscaled = scaler.fit_transform(Xtrain)
	Xtestscaled = scaler.transform(Xtest)
	return Xtrainscaled, Xtestscaled

# Scale values such that mean = 0, std dev. = 1; Ensures robustness for new data.
def standardize(Xtrain, Xtest):
	ss = SS()
	Xtrainscaled = ss.fit_transform(Xtrain)
	Xtestscaled = ss.transform(Xtest)
	return Xtrainscaled, Xtestscaled, ss

def micro_avg(y_test_multilabel, predictions):
	precision = precision_score(y_test_multilabel, predictions, average='micro')
	recall = recall_score(y_test_multilabel, predictions, average='micro')
	f1 = f1_score(y_test_multilabel, predictions, average='micro')

	print("::Micro-average::")
	print("Precision: {:.4f}, Recall: {:.4f}, F1-measure: {:.4f}".format(precision, recall, f1))
	print("\n\n")
	return precision, recall, f1

def macro_avg(y_test_multilabel, predictions):
	precision = precision_score(y_test_multilabel, predictions, average='macro')
	recall = recall_score(y_test_multilabel, predictions, average='macro')
	f1 = f1_score(y_test_multilabel, predictions, average='macro')

	print("\nMacro-average: ")
	print("Precision: {:.4f}, Recall: {:.4f}, F1-measure: {:.4f}".format(precision, recall, f1))
	return

def per_class_dist(ytest, ypred, classorder):
	perclass = classification_report(ytest, ypred)
	print("Per class classification report: ", perclass)
	precision, recall, fscore, support = score_multi(ytest, ypred, average="micro")
	print('micro-precision: {}'.format(precision))
	print('micro-recall: {}'.format(recall))
	print('micro-fscore: {}'.format(fscore))
	print('support: {}'.format(support))
	#print(classorder)
	return

def output_avg(total, ag_res1, ag_res2, fimp1, fimp2, auto_cmatrix, perf, auc_score, ff):
	print(auto_cmatrix)
	ff.write("-----------------Autogluon----------------\n")
	ff.write("Best model confusion matrix: \n")
	[tn,fp,fn,tp] = auto_cmatrix
	fpr = float(fp/(fp+tn)*100)
	ff.write("TN: "+str(tn)+" FP: "+str(fp)+" FN: "+str(fn)+" TP: "+str(tp)+"\n")
	ff.write("::Model performance on test data::\n")
	ff.write("AUC Score: "+str(auc_score)+"\n")
	ff.write("FPR: "+str(fpr)+"\n")
	ff.write("Performance summary: "+str(perf)+" \n")
	ff.write(str(ag_res1))
	if not fimp1 == None:
		ff.write("*Ft impo*\n")
		ff.write(str(fimp1.head(20))+"\n")
	ff.write("\n::Stacking & Weighted Ensembling of Models::\n")
	ff.write(str(ag_res2))
	if not fimp2 == None:
		ff.write("*Ft impo*\n")
		ff.write(str(fimp2.head(20))+"\n")
	ff.write("--------------------------------------------\n")
	ff.close()
	return

In [ ]:
def test_main(xtest, ytest, pred, testdf, traindf, calcftimpo=False):
	modelperf = pred.leaderboard(testdf, silent= True)
	print("[*]Model performance breakdown on Test data:")
	print(modelperf)
	ypred = pred.predict(xtest)
	ypredproba = pred.predict_proba(xtest)
	perf = pred.evaluate_predictions(y_true=ytest, y_pred=ypred, auxiliary_metrics= True)
	print("[*]Predictions: ", ypred)
	print("[*]Confidence in predictions:\n")
	print(pd.DataFrame(ypredproba, columns=pred.class_labels))
	# Each model score
	print("Perf: ", perf)
	print(classification_report(y_test, ypred, output_dict=True))

	print("Getting confusion matrix.....")
	cmatrix = confusion_matrix(ytest, ypred).ravel().tolist()
	print(cmatrix)
	auc_score = roc_auc_score(ytest, ypredproba.iloc[:, 1])
	print("AUC score for best model: ", auc_score)

	if calcftimpo:
		ftimpo = None
		ftimpo = pred.feature_importance(traindf)
		print("Feature Importance on test data: ", ftimpo)
	else:
		ftimpo = None
	bestmodel = pred.model_best
	# Find mistakes
	misclassified = xtest[ypred != ytest].copy()
	misclassified['true_label'] = ytest[ypred != ytest]
	misclassified['predicted_label'] = ypred[ypred != ytest]
	return modelperf, ftimpo, cmatrix, ypredproba, bestmodel, perf, auc_score, misclassified

def test_stack(xtest, ytest, predstack, testdf, traindf, calcftimpo=False):
	ypred = predstack.predict(xtest)
	ypredproba = predstack.predict_proba(xtest)
	perf = predstack.evaluate_predictions(y_true=ytest, y_pred=ypred, auxiliary_metrics= True)
	print("[*]Predictions: ", ypred)
	test_perf = predstack.leaderboard(testdf, silent=True)
	print("$$$$$$$$ RESULT STACKING $$$$$$$$\n", test_perf)
	ftimpo = None
	if calcftimpo:
		ftimpo = predstack.feature_importance(traindf)
		print("Feature Importance on test data: ", ftimpo)
	auc_score = roc_auc_score(ytest, ypredproba.iloc[:, 1])
	cmatrix = confusion_matrix(ytest, ypred).ravel().tolist()
	print("Confusion matrix stacked: ", cmatrix)
	print("AUC using stacked model: ", auc_score)
	return test_perf, ftimpo, cmatrix, auc_score

In [ ]:
def train_main(dataf, valdf, targetcol):
	agdir = os.getcwd()+'/AGmodels/'
	#dir = agdir+"/"+str(malinst)+"_"+str(hostfts)+"/"
	if not os.path.exists(agdir):
		os.system("mkdir "+agdir)

	predictor = task(label=targetcol, path=agdir, eval_metric='f1').fit( train_data=dataf, tuning_data=valdf, verbosity=3)
	return predictor

# Multi layer stacking takes predictions of base models and feeds to stack models
# AG will auto choose k= 10 fold cv, n=20 bagging repeats,
# L: 2 layers of models in stack followed by weighted-ensemble (higher weight for the model that performed well);
# Aggregate model predictions based on model weights and produce final prediction
def train_multilayerstacking(traindf, target):
	agdir_stack = os.getcwd()+'/AGmodels/stacked/' #+str(malinst)+"_"+str(hostfts)+"/"
	if not os.path.exists(agdir_stack):
		os.system("mkdir "+agdir_stack)
	predstack = task(label=target, path=agdir_stack, eval_metric='f1').fit(train_data= traindf, auto_stack=True, verbosity=3)
	return predstack


In [ ]:
relayed_fts_path = '/content/features_lim_relayed_train_rtt.csv'
background_fts_path = '/content/features_lim_background_train_rtt.csv'
relayed_feats_train=pd.read_csv(relayed_fts_path)
background_feats_train=pd.read_csv(background_fts_path)
print(background_feats_train.shape)
print(relayed_feats_train.shape)

(79246, 169)
(13438, 169)


In [ ]:
relayed_corr_fts_train_path = '/content/feature_corr__relayed_train.csv'
background_corr_fts_train_path = '/content/feature_corr__background_train.csv'
relayed_corr_fts_test_path = '/content/feature_corr__relayed_test.csv'
background_corr_fts_test_path = '/content/feature_corr__background_test.csv'
relayed_corr_fts_val_path = '/content/feature_corr__relayed_val.csv'
background_corr_fts_val_path = '/content/feature_corr__background_val.csv'

relayed_corr_feats_train=pd.read_csv(relayed_corr_fts_train_path)
background_corr_feats_train=pd.read_csv(background_corr_fts_train_path)
relayed_corr_feats_test=pd.read_csv(relayed_corr_fts_test_path)
background_corr_feats_test=pd.read_csv(background_corr_fts_test_path)
relayed_corr_feats_val=pd.read_csv(relayed_corr_fts_val_path)
background_corr_feats_val=pd.read_csv(background_corr_fts_val_path)



In [ ]:
relayed_corr_feats_train.shape

(18941, 3)

In [ ]:
foldtotal = 10
"""
relayed_fts_path = '/content/features_lim_relayed_train_rtt.csv'
background_fts_path = '/content/features_lim_background_train_rtt.csv'
relayed_fts_test_path = '/content/features_lim_relayed_test_rtt.csv'
background_fts_test_path = '/content/features_lim_background_test_rtt.csv'
relayed_feats_val_path='/content/features_lim_relayed_val_rtt.csv'
background_feats_val_path='/content/features_lim_background_val_rtt.csv'
"""
relayed_fts_path='/content/all_relayed_train.csv'
background_fts_path='/content/all_background_train.csv'
relayed_fts_test_path='/content/all_relayed_test.csv'
background_fts_test_path='/content/all_background_test.csv'
relayed_feats_val_path='/content/all_relayed_val.csv'
background_feats_val_path='/content/all_background_val.csv'

relayed_feats_train=pd.read_csv(relayed_fts_path)
background_feats_train=pd.read_csv(background_fts_path)

relayed_feats_test=pd.read_csv(relayed_fts_test_path)
background_feats_test=pd.read_csv(background_fts_test_path)

relayed_feats_val=pd.read_csv(relayed_feats_val_path)
background_feats_val=pd.read_csv(background_feats_val_path)



background_feats_train['label']=0
relayed_feats_train['label']=1
train=pd.concat([relayed_feats_train,background_feats_train],ignore_index=True)
train=train.drop(columns=['pcap_nb'])
train=train.drop(columns=['conn'])

train=train.drop(columns=['mean_bytes_sent'])
train=train.drop(columns=['mean_vol_total_pkts'])

#train=train.drop(columns=['std_order_out'])
train=train.sample(frac=1, random_state=42)
train=train.reset_index(drop=True)
train_original = train.copy()
std_devs = train.drop(columns=['label']).std()
features_to_keep = std_devs[std_devs != 0].index
train_filtered = train[features_to_keep].copy()
train_filtered['label'] = train['label']
# # Separate classes
# class_0 = train_filtered[train_filtered['label'] == 0]
# class_1 = train_filtered[train_filtered['label'] == 1]

# # Perform t-tests and filter features based on p-value threshold
# p_value_threshold = 0.05
# selected_features = []

# for feature in features_to_keep:
#     class_0_values = class_0[feature]
#     class_1_values = class_1[feature]

#     t_stat, p_value = ttest_ind(class_0_values, class_1_values)

#     if p_value <= p_value_threshold:
#         selected_features.append(feature)

# # Create a new DataFrame with only the selected features
# train_filtered = train_filtered[selected_features + ['label']]

# Display the resulting DataFrame
relayed_feats_val['label'] = 1
background_feats_val['label'] = 0
val_data=pd.concat([relayed_feats_val,background_feats_val])
val_data=val_data.drop(columns=['pcap_nb'])
val_data=val_data.drop(columns=['conn'])

val_data=val_data.drop(columns=['mean_bytes_sent'])
val_data=val_data.drop(columns=['mean_vol_total_pkts'])
#val_data=val_data.drop(columns=['std_order_out'])
val=val_data.sample(frac=1, random_state=42)
val=val.reset_index(drop=True)
val_filtered = val[features_to_keep].copy()
val_filtered['label'] = val['label']
# val_filtered=val_filtered[selected_features+['label']]


background_feats_test['label']=0
relayed_feats_test['label']=1
test=pd.concat([relayed_feats_test,background_feats_test],ignore_index=True)
test=test.sample(frac=1, random_state=42)
test_original = test.copy()
test=test.drop(columns=['pcap_nb'])
test=test.drop(columns=['conn'])
test=test.drop(columns=['mean_bytes_sent'])
test=test.drop(columns=['mean_vol_total_pkts'])
#test=test.drop(columns=['std_order_out'])
test=test.reset_index(drop=True)
test_filtered = test[features_to_keep].copy()
test_filtered['label'] = test['label']
# test_filtered=test_filtered[selected_features+['label']]

majority_class = train_filtered[train_filtered['label'] == 0]
minority_class = train_filtered[train_filtered['label'] == 1]

desired_majority_size = int(len(minority_class) * 6)

majority_downsampled = resample(majority_class,
                                replace=False,
                                n_samples=desired_majority_size,
                                random_state=42)
print(majority_downsampled.shape)
print(minority_class.shape)
# Combine the adjusted majority and minority classes
train_balanced = pd.concat([majority_downsampled, minority_class])
train_balanced = train_balanced.dropna()

(48066, 87)
(8011, 87)


In [ ]:
print(background_feats_train.shape)
print(relayed_feats_train.shape)
print(background_feats_test.shape)
print(relayed_feats_test.shape)
print(background_feats_val.shape)
print(relayed_feats_val.shape)

(674654, 167)
(8011, 167)
(199659, 167)
(3355, 167)
(194685, 167)
(2804, 167)


In [ ]:
nan_columns = train_balanced.isna().any()
print("Columns with NaN values:")
print(nan_columns[nan_columns])


Columns with NaN values:
Series([], dtype: bool)


In [ ]:
# Training binary classifiers: 8 base models, 2 DL models
predictor = train_main(train_balanced,val_filtered, "label")

Verbosity: 3 (Detailed Logging)
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.11.11
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun 27 21:05:47 UTC 2024
CPU Count:          2
GPU Count:          1
Memory Avail:       4.50 GB / 12.67 GB (35.5%)
Disk Space Avail:   79.28 GB / 112.64 GB (70.4%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='experimental' : New in v1.2: Pre-trained foundation model + parallel fits. The absolute best accuracy without consideration for inference speed. Does not support GPU.
	presets='best'         : Maximize accuracy. Recommended for most users. Use in competitions and benchmarks.
	presets='high'         : Strong accuracy wi

[50]	valid_set's binary_logloss: 0.0694928	valid_set's f1: 0.610241
[100]	valid_set's binary_logloss: 0.0465925	valid_set's f1: 0.662892
[150]	valid_set's binary_logloss: 0.0356972	valid_set's f1: 0.717657
[200]	valid_set's binary_logloss: 0.0300069	valid_set's f1: 0.745907
[250]	valid_set's binary_logloss: 0.0264833	valid_set's f1: 0.767314
[300]	valid_set's binary_logloss: 0.0240392	valid_set's f1: 0.780975
[350]	valid_set's binary_logloss: 0.0222764	valid_set's f1: 0.790932
[400]	valid_set's binary_logloss: 0.0209781	valid_set's f1: 0.797555
[450]	valid_set's binary_logloss: 0.0197911	valid_set's f1: 0.806015
[500]	valid_set's binary_logloss: 0.0189122	valid_set's f1: 0.808446
[550]	valid_set's binary_logloss: 0.0181771	valid_set's f1: 0.812974
[600]	valid_set's binary_logloss: 0.0176249	valid_set's f1: 0.817629
[650]	valid_set's binary_logloss: 0.0171249	valid_set's f1: 0.820107
[700]	valid_set's binary_logloss: 0.016681	valid_set's f1: 0.823404
[750]	valid_set's binary_logloss: 0.

Saving /content/AGmodels/models/LightGBMXT/model.pkl
Saving /content/AGmodels/utils/attr/LightGBMXT/y_pred_proba_val.pkl
	0.8513	 = Validation score   (f1)
	852.43s	 = Training   runtime
	82.95s	 = Validation runtime
	2380.7	 = Inference  throughput (rows/s | 197489 batch size)
Saving /content/AGmodels/models/trainer.pkl
Fitting model: LightGBM ...
	Fitting LightGBM with 'num_gpus': 0, 'num_cpus': 1
	Fitting 10000 rounds... Hyperparameters: {'learning_rate': 0.05}


[50]	valid_set's binary_logloss: 0.0522246	valid_set's f1: 0.690909
[100]	valid_set's binary_logloss: 0.0315271	valid_set's f1: 0.741739
[150]	valid_set's binary_logloss: 0.0242199	valid_set's f1: 0.781981
[200]	valid_set's binary_logloss: 0.0205174	valid_set's f1: 0.807182
[250]	valid_set's binary_logloss: 0.0183552	valid_set's f1: 0.821603
[300]	valid_set's binary_logloss: 0.0169284	valid_set's f1: 0.828065
[350]	valid_set's binary_logloss: 0.0159381	valid_set's f1: 0.835196
[400]	valid_set's binary_logloss: 0.0151455	valid_set's f1: 0.839857
[450]	valid_set's binary_logloss: 0.0145236	valid_set's f1: 0.845804
[500]	valid_set's binary_logloss: 0.0140782	valid_set's f1: 0.847761
[550]	valid_set's binary_logloss: 0.0136835	valid_set's f1: 0.851285
[600]	valid_set's binary_logloss: 0.0134532	valid_set's f1: 0.853215
[650]	valid_set's binary_logloss: 0.0132339	valid_set's f1: 0.85416
[700]	valid_set's binary_logloss: 0.0130623	valid_set's f1: 0.856328
[750]	valid_set's binary_logloss: 0.

Saving /content/AGmodels/models/LightGBM/model.pkl
Saving /content/AGmodels/utils/attr/LightGBM/y_pred_proba_val.pkl
	0.8734	 = Validation score   (f1)
	320.23s	 = Training   runtime
	32.53s	 = Validation runtime
	6070.4	 = Inference  throughput (rows/s | 197489 batch size)
Saving /content/AGmodels/models/trainer.pkl
Fitting model: RandomForestGini ...
	Fitting RandomForestGini with 'num_gpus': 0, 'num_cpus': 2
Saving /content/AGmodels/models/RandomForestGini/model.pkl
Saving /content/AGmodels/utils/attr/RandomForestGini/y_pred_proba_val.pkl
	0.7608	 = Validation score   (f1)
	60.54s	 = Training   runtime
	5.63s	 = Validation runtime
	35056.7	 = Inference  throughput (rows/s | 197489 batch size)
Saving /content/AGmodels/models/trainer.pkl
Fitting model: RandomForestEntr ...
	Fitting RandomForestEntr with 'num_gpus': 0, 'num_cpus': 2
Saving /content/AGmodels/models/RandomForestEntr/model.pkl
Saving /content/AGmodels/utils/attr/RandomForestEntr/y_pred_proba_val.pkl
	0.7637	 = Validation 

...

Saving /content/AGmodels/models/CatBoost/model.pkl
Saving /content/AGmodels/utils/attr/CatBoost/y_pred_proba_val.pkl
	0.8653	 = Validation score   (f1)
	502.83s	 = Training   runtime
	1.73s	 = Validation runtime
	114214.0	 = Inference  throughput (rows/s | 197489 batch size)
Saving /content/AGmodels/models/trainer.pkl
Fitting model: ExtraTreesGini ...
	Fitting ExtraTreesGini with 'num_gpus': 0, 'num_cpus': 2
Saving /content/AGmodels/models/ExtraTreesGini/model.pkl
Saving /content/AGmodels/utils/attr/ExtraTreesGini/y_pred_proba_val.pkl
	0.7399	 = Validation score   (f1)
	15.2s	 = Training   runtime
	13.42s	 = Validation runtime
	14717.1	 = Inference  throughput (rows/s | 197489 batch size)
Saving /content/AGmodels/models/trainer.pkl
Fitting model: ExtraTreesEntr ...
	Fitting ExtraTreesEntr with 'num_gpus': 0, 'num_cpus': 2
Saving /content/AGmodels/models/ExtraTreesEntr/model.pkl
Saving /content/AGmodels/utils/attr/ExtraTreesEntr/y_pred_proba_val.pkl
	0.7393	 = Validation score   (f1)
	1

[0]	validation_0-logloss:0.21590	validation_0-_f1:-0.00000
[50]	validation_0-logloss:0.03852	validation_0-_f1:-0.70726
[100]	validation_0-logloss:0.02481	validation_0-_f1:-0.78290
[150]	validation_0-logloss:0.02024	validation_0-_f1:-0.80792
[200]	validation_0-logloss:0.01802	validation_0-_f1:-0.81982
[250]	validation_0-logloss:0.01612	validation_0-_f1:-0.83357
[300]	validation_0-logloss:0.01514	validation_0-_f1:-0.84164
[350]	validation_0-logloss:0.01446	validation_0-_f1:-0.84442
[400]	validation_0-logloss:0.01403	validation_0-_f1:-0.84899
[450]	validation_0-logloss:0.01371	validation_0-_f1:-0.85308
[500]	validation_0-logloss:0.01356	validation_0-_f1:-0.85524
[550]	validation_0-logloss:0.01349	validation_0-_f1:-0.85592
[600]	validation_0-logloss:0.01324	validation_0-_f1:-0.85801
[650]	validation_0-logloss:0.01321	validation_0-_f1:-0.85742
[700]	validation_0-logloss:0.01315	validation_0-_f1:-0.85841
[750]	validation_0-logloss:0.01317	validation_0-_f1:-0.85959
[800]	validation_0-logloss:

Saving /content/AGmodels/models/XGBoost/model.pkl
Saving /content/AGmodels/utils/attr/XGBoost/y_pred_proba_val.pkl
	0.8675	 = Validation score   (f1)
	1687.86s	 = Training   runtime
	58.92s	 = Validation runtime
	3351.7	 = Inference  throughput (rows/s | 197489 batch size)
Saving /content/AGmodels/models/trainer.pkl
Fitting model: NeuralNetTorch ...
	Fitting NeuralNetTorch with 'num_gpus': 0, 'num_cpus': 1
Tabular Neural Network treats features as the following types:
{
    "continuous": [
        "mean_bytes_recv",
        "median_bytes_recv",
        "mode_bytes_recv",
        "avg_in",
        "avg_out",
        "avg_total",
        "nb_pkts_in",
        "nb_pkts_out",
        "nb_pkts_in_l30",
        "nb_pkts_out_l30",
        "std_pkt_conc_out20",
        "avg_pkt_conc_out20",
        "avg_order_in",
        "avg_order_out",
        "std_order_in",
        "std_order_out",
        "maxconc",
        "perc_in",
        "perc_out",
        "sum_altconc",
        "altconc_8",
      

[50]	valid_set's binary_logloss: 0.0612614	valid_set's f1: 0.761173


Saving /content/AGmodels/models/LightGBMLarge/model.pkl
Saving /content/AGmodels/utils/attr/LightGBMLarge/y_pred_proba_val.pkl
	0.789	 = Validation score   (f1)
	15.45s	 = Training   runtime
	0.67s	 = Validation runtime
	292664.1	 = Inference  throughput (rows/s | 197489 batch size)
Saving /content/AGmodels/models/trainer.pkl
Loading: /content/AGmodels/utils/attr/ExtraTreesGini/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/RandomForestGini/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/NeuralNetFastAI/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/CatBoost/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/RandomForestEntr/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/LightGBMLarge/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/KNeighborsUnif/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/ExtraTreesEntr/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/LightGBM/y_pred_proba_val.pkl
Loading: /cont

In [ ]:
#background_feats_test_path='/content/features_lim_background_bg.csv'
#background_feats_test=pd.read_csv(background_feats_test_path)
#background_feats_test['label']=0
x_test = test_filtered.iloc[:,:-1].copy()
y_test = test_filtered.iloc[:,-1].copy()
print("###################~Testing Trained Models ############################")
#res1, fimp1, cmatrix, ypred_proba, bestmodel, perf, auc_score = test_main(x_val, y_val, predictor, val_data, train)
# Uncomment for test results with feature importance (longer run time)
res1, fimp1, cmatrix, ypred_proba, bestmodel, perf, auc_score, mistakes = test_main(x_test, y_test, predictor, test_filtered, train_filtered,True)

#print("####################Stacking & Weighted Ensemble Testing###########################")
#res2, fimp2, cmatrixstacked, aucstacked = test_stack(x_val, y_val, predstack, val_data, train)
# With feature importance
#res2, fimp2, cmatrixstacked, aucstacked = test_stack(x_test, y_test, predstack, test, train, True)

###################~Testing Trained Models ############################


Loading: /content/AGmodels/models/KNeighborsUnif/model.pkl
Loading: /content/AGmodels/models/KNeighborsDist/model.pkl
Loading: /content/AGmodels/models/LightGBMXT/model.pkl
Loading: /content/AGmodels/models/LightGBM/model.pkl
Loading: /content/AGmodels/models/RandomForestGini/model.pkl
Loading: /content/AGmodels/models/RandomForestEntr/model.pkl
Loading: /content/AGmodels/models/CatBoost/model.pkl
Loading: /content/AGmodels/models/ExtraTreesGini/model.pkl
Loading: /content/AGmodels/models/ExtraTreesEntr/model.pkl
Loading: /content/AGmodels/models/NeuralNetFastAI/model.pkl
Loading: /content/AGmodels/models/NeuralNetFastAI/model-internals.pkl
Loading: /content/AGmodels/models/XGBoost/model.pkl
Loading: /content/AGmodels/models/NeuralNetTorch/model.pkl
Loading: /content/AGmodels/models/LightGBMLarge/model.pkl
Loading: /content/AGmodels/models/WeightedEnsemble_L2/model.pkl
Loading: /content/AGmodels/models/CatBoost/model.pkl


[*]Model performance breakdown on Test data:
                  model  score_test  score_val eval_metric  pred_time_test  \
0   WeightedEnsemble_L2    0.918266   0.901501          f1      137.379151   
1              LightGBM    0.903859   0.873367          f1       32.212578   
2              CatBoost    0.902341   0.865264          f1        1.779459   
3               XGBoost    0.902004   0.867520          f1       53.482446   
4            LightGBMXT    0.891349   0.851292          f1       86.180647   
5        NeuralNetTorch    0.848510   0.806590          f1        2.812827   
6      RandomForestEntr    0.836732   0.763722          f1        5.015449   
7      RandomForestGini    0.835204   0.760751          f1        7.992898   
8        ExtraTreesGini    0.814782   0.739916          f1       14.312254   
9        ExtraTreesEntr    0.812319   0.739281          f1       12.526668   
10        LightGBMLarge    0.792447   0.788959          f1        0.427033   
11      NeuralNetFa

Loading: /content/AGmodels/models/KNeighborsUnif/model.pkl
Loading: /content/AGmodels/models/LightGBM/model.pkl
Loading: /content/AGmodels/models/LightGBMLarge/model.pkl
Loading: /content/AGmodels/models/NeuralNetFastAI/model.pkl
Loading: /content/AGmodels/models/NeuralNetFastAI/model-internals.pkl
Loading: /content/AGmodels/models/NeuralNetTorch/model.pkl
Loading: /content/AGmodels/models/WeightedEnsemble_L2/model.pkl
Loading: /content/AGmodels/models/CatBoost/model.pkl
Loading: /content/AGmodels/models/KNeighborsUnif/model.pkl
Loading: /content/AGmodels/models/LightGBM/model.pkl
Loading: /content/AGmodels/models/LightGBMLarge/model.pkl
Loading: /content/AGmodels/models/NeuralNetFastAI/model.pkl
Loading: /content/AGmodels/models/NeuralNetFastAI/model-internals.pkl
Loading: /content/AGmodels/models/NeuralNetTorch/model.pkl
Loading: /content/AGmodels/models/WeightedEnsemble_L2/model.pkl
Skipping roc_auc because no prediction probabilities are available to score.


[*]Predictions:  0         0
1         0
2         0
3         0
4         0
         ..
203009    0
203010    0
203011    0
203012    0
203013    0
Name: label, Length: 203014, dtype: int64
[*]Confidence in predictions:

               0         1
0       0.989781  0.010219
1       0.986906  0.013094
2       0.988966  0.011034
3       0.986320  0.013679
4       0.985688  0.014312
...          ...       ...
203009  0.987758  0.012242
203010  0.989127  0.010873
203011  0.962526  0.037474
203012  0.987323  0.012677
203013  0.866859  0.133141

[203014 rows x 2 columns]
Perf:  {'f1': 0.9182655990330866, 'accuracy': 0.9973351591515857, 'balanced_accuracy': 0.9523426495826651, 'mcc': 0.9170009218319413, 'precision': 0.9310661764705882, 'recall': 0.9058122205663189}
{'0': {'precision': 0.9984180225281603, 'recall': 0.9988730785990113, 'f1-score': 0.9986454987243653, 'support': 199659.0}, '1': {'precision': 0.9310661764705882, 'recall': 0.9058122205663189, 'f1-score': 0.9182655990330866, 'supp

These features in provided data are not utilized by the predictor and will be ignored: ['min_per_sec', 'alt_per_sec_9', 'alt_per_sec_10', 'alt_per_sec_18', 'alt_per_sec_19', 'conc_2', 'conc_3']


AUC score for best model:  0.9986378533969718


Loading: /content/AGmodels/models/WeightedEnsemble_L2/model.pkl
Computing feature importance via permutation shuffling for 79 features using 5000 rows with 5 shuffle sets...
Loading: /content/AGmodels/models/CatBoost/model.pkl
Loading: /content/AGmodels/models/KNeighborsUnif/model.pkl
Loading: /content/AGmodels/models/LightGBM/model.pkl
Loading: /content/AGmodels/models/LightGBMLarge/model.pkl
Loading: /content/AGmodels/models/NeuralNetFastAI/model.pkl
Loading: /content/AGmodels/models/NeuralNetFastAI/model-internals.pkl
Loading: /content/AGmodels/models/NeuralNetTorch/model.pkl
Loading: /content/AGmodels/models/WeightedEnsemble_L2/model.pkl
	1897.91s	= Expected runtime (379.58s per shuffle set)
Loading: /content/AGmodels/models/CatBoost/model.pkl
Loading: /content/AGmodels/models/KNeighborsUnif/model.pkl
Loading: /content/AGmodels/models/LightGBM/model.pkl
Loading: /content/AGmodels/models/LightGBMLarge/model.pkl
Loading: /content/AGmodels/models/NeuralNetFastAI/model.pkl
Loading: /co

Feature Importance on test data:                            importance    stddev   p_value  n  p99_high  \
rtt                         0.222965  0.043730  0.000169  5  0.313005   
std_total                   0.199020  0.025621  0.000032  5  0.251775   
mean_bytes_recv             0.184949  0.065894  0.001645  5  0.320627   
75th_percentile_out         0.161519  0.011386  0.000003  5  0.184964   
std_in                      0.153576  0.018885  0.000027  5  0.192460   
...                              ...       ...       ... ..       ...   
nb_pkts_out                -0.018775  0.016146  0.969982  5  0.014469   
perc_out                   -0.019569  0.020076  0.952611  5  0.021767   
nb_pkts_in                 -0.020714  0.020125  0.958598  5  0.020724   
75th_percentile_out_time   -0.023462  0.019061  0.974375  5  0.015784   
nb_pkts_in_l30             -0.024114  0.024945  0.951644  5  0.027247   

                           p99_low  
rtt                       0.132925  
std_total      

In [ ]:
cmatrix

[213349, 451, 124, 2710]

In [ ]:
TP = cmatrix[0]
FN = cmatrix[1]
FP = cmatrix[2]
TN = cmatrix[3]

# Calculate FPR and FNR
FPR = (FP / (FP + TN))*100 if (FP + TN) > 0 else 0
FNR = (FN / (FN + TP))*100 if (FN + TP) > 0 else 0

print(f"False Positive Rate (FPR): {FPR:.4f} %")
print(f"False Negative Rate (FNR): {FNR:.4f} %")

False Positive Rate (FPR): 4.3754 %
False Negative Rate (FNR): 0.2109 %


In [ ]:
fimp1.head(20)

,importance,stddev,p_value,n,p99_high,p99_low
rtt,0.216865,0.043062,0.000177,5,0.305532,0.128199
std_in,0.191403,0.027050,0.000047,5,0.247098,0.135707
75th_percentile_out,0.130059,0.026502,0.000196,5,0.184627,0.075490
std_total,0.126373,0.023400,0.000135,5,0.174555,0.078192
mean_bytes_recv,0.080287,0.013757,0.000100,5,0.108614,0.051960
100th_percentile_in_time,0.076284,0.019167,0.000441,5,0.115749,0.036818
std_out,0.055497,0.015562,0.000670,5,0.087540,0.023454
75th_percentile_in,0.038297,0.013516,0.001589,5,0.066127,0.010467
max_out,0.037319,0.017993,0.004875,5,0.074368,0.000271
avg_order_in,0.025427,0.009439,0.001913,5,0.044861,0.005993


In [ ]:
selected_columns =  test_original.loc[mistakes.index, ['conn', 'pcap_nb']]

selected_columns_copy = selected_columns.copy()
selected_columns_copy.to_csv('selected_conn_pcap.csv', index=False)

print(selected_columns_copy)

                   conn  \
124    normal_conn_3762   
284     normal_conn_470   
655    normal_conn_1740   
1126   normal_conn_2987   
1321   normal_conn_1828   
...                 ...   
94146    normal_conn_89   
94303  normal_conn_1371   
94680   normal_conn_324   
94689   normal_conn_352   
94886   normal_conn_902   

                                                                                                  pcap_nb  
124    /Users/mounarabhi/Desktop/ProxyTrafficAnalysis/data/processed_new/mixed_27_12_2024_00_21_25_medium  
284       /Users/mounarabhi/Desktop/ProxyTrafficAnalysis/data/processed_new/mixed_27_12_2024_22_03_46_low  
655       /Users/mounarabhi/Desktop/ProxyTrafficAnalysis/data/processed_new/mixed_28_12_2024_23_37_37_low  
1126      /Users/mounarabhi/Desktop/ProxyTrafficAnalysis/data/processed_new/mixed_28_12_2024_17_14_13_low  
1321   /Users/mounarabhi/Desktop/ProxyTrafficAnalysis/data/processed_new/mixed_29_12_2024_09_16_06_medium  
...                        

In [ ]:
predstack = train_multilayerstacking(train_balanced, "label")

Verbosity: 3 (Detailed Logging)
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.10.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun 27 21:05:47 UTC 2024
CPU Count:          2
GPU Count:          1
Memory Avail:       9.65 GB / 12.67 GB (76.2%)
Disk Space Avail:   78.53 GB / 112.64 GB (69.7%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='experimental' : New in v1.2: Pre-trained foundation model + parallel fits. The absolute best accuracy without consideration for inference speed. Does not support GPU.
	presets='best'         : Maximize accuracy. Recommended for most users. Use in competitions and benchmarks.
	presets='high'         : Strong accuracy wi

In [ ]:
res2, fimp2, cmatrixstacked, aucstacked = test_stack(x_test, y_test, predstack, test, train, True)

Loading: /content/AGmodels/stacked/models/CatBoost_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/ExtraTreesGini_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/LightGBMXT_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/LightGBM_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/NeuralNetFastAI_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/NeuralNetTorch_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/RandomForestEntr_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/XGBoost_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/WeightedEnsemble_L2/model.pkl
Loading: /content/AGmodels/stacked/models/CatBoost_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/ExtraTreesGini_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/LightGBMXT_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/LightGBM_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/NeuralNetFastAI_BAG_L1/model.pkl
L

[*]Predictions:  0        0
1        0
2        0
3        0
4        0
        ..
56503    0
56504    0
56505    0
56506    0
56507    0
Name: label, Length: 56508, dtype: int64


Loading: /content/AGmodels/stacked/models/KNeighborsDist_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/LightGBMXT_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/LightGBM_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/RandomForestGini_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/RandomForestEntr_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/CatBoost_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/ExtraTreesGini_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/ExtraTreesEntr_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/NeuralNetFastAI_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/XGBoost_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/NeuralNetTorch_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/LightGBMLarge_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/WeightedEnsemble_L2/model.pkl
These features in provided data are not utilized by the pre

$$$$$$$$ RESULT STACKING $$$$$$$$
                       model  score_test  score_val eval_metric  \
0       WeightedEnsemble_L2    0.695043   0.940127          f1   
1           CatBoost_BAG_L1    0.691383   0.929374          f1   
2           LightGBM_BAG_L1    0.677013   0.934976          f1   
3         LightGBMXT_BAG_L1    0.663781   0.931961          f1   
4            XGBoost_BAG_L1    0.659696   0.933180          f1   
5      LightGBMLarge_BAG_L1    0.655582   0.926634          f1   
6   RandomForestGini_BAG_L1    0.649174   0.901542          f1   
7   RandomForestEntr_BAG_L1    0.643987   0.902540          f1   
8     ExtraTreesGini_BAG_L1    0.641267   0.880624          f1   
9     ExtraTreesEntr_BAG_L1    0.640671   0.882937          f1   
10    NeuralNetTorch_BAG_L1    0.560327   0.912933          f1   
11   NeuralNetFastAI_BAG_L1    0.460091   0.854201          f1   
12    KNeighborsDist_BAG_L1    0.382382   0.769904          f1   
13    KNeighborsUnif_BAG_L1    0.374875  

Loading: /content/AGmodels/stacked/models/ExtraTreesGini_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/LightGBMXT_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/LightGBM_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/NeuralNetFastAI_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/NeuralNetTorch_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/RandomForestEntr_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/XGBoost_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/WeightedEnsemble_L2/model.pkl
	1589.12s	= Expected runtime (317.82s per shuffle set)
Loading: /content/AGmodels/stacked/models/CatBoost_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/ExtraTreesGini_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/LightGBMXT_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/LightGBM_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/NeuralNetFastAI_BAG_L1/model.pkl
Loading: /cont

Feature Importance on test data:                         importance    stddev   p_value  n  p99_high   p99_low
mean_vol_total_pkts      0.364305  0.044876  0.000027  5  0.456705  0.271905
mean_bytes_sent          0.200338  0.022235  0.000018  5  0.246119  0.154556
75th_percentile_out      0.078204  0.030159  0.002200  5  0.140302  0.016106
std_in                   0.065268  0.034267  0.006532  5  0.135824 -0.005287
gap_between_conns        0.057276  0.048421  0.028640  5  0.156975 -0.042423
median_vol_total_pkts    0.048106  0.041815  0.030910  5  0.134204 -0.037993
75th_percentile_total    0.036232  0.017125  0.004549  5  0.071493  0.000971
mean_bytes_recv          0.032994  0.010876  0.001233  5  0.055388  0.010600
max_total                0.029003  0.023453  0.025288  5  0.077292 -0.019286
std_order_in             0.026485  0.030531  0.062206  5  0.089350 -0.036379
avg_order_in             0.021892  0.013643  0.011503  5  0.049983 -0.006200
std_total                0.017134  0.01831

In [ ]:
ff = open("./BinaryTraining.score", "w+")
output_avg(foldtotal, res1, res2, fimp1, fimp2, cmatrix, perf,auc_score, ff)

NameError: name 'res2' is not defined